# Notebook 01: Gaussian Distribution Basics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChunLI-666/3DGS-from-scratch/blob/develop/notebooks/phase1/01_gaussian_basics.ipynb)

---

## Learning Objectives

By the end of this notebook, you will:
1. Understand 1D, 2D, and 3D Gaussian distributions
2. Know how mean and variance/covariance affect the distribution shape
3. Visualize Gaussians interactively
4. Build intuition for why Gaussians are used in 3DGS

**Estimated Time**: 45 minutes

**Prerequisites**: Notebook 00 (Environment Setup)

---

## Setup

In [ ]:
import os
import sys

# Environment setup for Colab
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('3DGS-from-scratch'):
        !git clone https://github.com/ChunLI-666/3DGS-from-scratch.git
    os.chdir('3DGS-from-scratch')
    !pip install -q plotly ipywidgets

# Add src to path
for path in ['../../src', '../src', './src', '../../']:
    full_path = os.path.abspath(path)
    if os.path.exists(os.path.join(full_path, 'gaussian')) or os.path.exists(os.path.join(full_path, 'src', 'gaussian')):
        if 'src' in full_path:
            sys.path.insert(0, full_path)
        else:
            sys.path.insert(0, os.path.join(full_path, 'src'))
        break

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from mpl_toolkits.mplot3d import Axes3D
from scipy import stats

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

print("Setup complete!")

## 1. Why Gaussians?

Before diving into the math, let's understand **why** 3D Gaussian Splatting uses Gaussians to represent scenes.

### Key Properties of Gaussians:

1. **Smooth and Continuous**: Gaussians have smooth falloff, creating natural-looking blends
2. **Mathematically Tractable**: Easy to project, transform, and differentiate
3. **Compact Representation**: Fully described by just mean and covariance
4. **Natural for Optimization**: Smooth gradients for gradient-based optimization
5. **Efficient Rendering**: Can be efficiently "splatted" onto the image plane

### Comparison with Other Representations:

| Representation | Pros | Cons |
|---------------|------|------|
| **Points** | Simple | Sharp, no natural blending |
| **Voxels** | Regular grid | Memory intensive, blocky |
| **Meshes** | Well-established | Hard to optimize, topology issues |
| **NeRF (MLP)** | Flexible | Slow rendering, implicit |
| **Gaussians** | Smooth, fast, explicit | More parameters than points |

## 2. 1D Gaussian Distribution

Let's start with the simplest case: a 1D Gaussian (normal distribution).

### Mathematical Definition

A 1D Gaussian probability density function (PDF) is defined as:

$$G(x) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

Where:
- $\mu$ (mu) = **mean** - the center of the distribution
- $\sigma$ (sigma) = **standard deviation** - controls the width
- $\sigma^2$ = **variance**

In [ ]:
def gaussian_1d(x, mu, sigma):
    """
    1D Gaussian probability density function.
    
    Args:
        x: Input values
        mu: Mean (center)
        sigma: Standard deviation (width)
    
    Returns:
        Gaussian values at x
    """
    coefficient = 1 / (np.sqrt(2 * np.pi) * sigma)
    exponent = -((x - mu) ** 2) / (2 * sigma ** 2)
    return coefficient * np.exp(exponent)

# Visualize 1D Gaussians with different parameters
x = np.linspace(-6, 6, 500)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Effect of mean (mu)
ax1 = axes[0]
for mu in [-2, 0, 2]:
    y = gaussian_1d(x, mu, sigma=1.0)
    ax1.plot(x, y, linewidth=2, label=f'$\mu$ = {mu}')
ax1.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
ax1.set_xlabel('x', fontsize=12)
ax1.set_ylabel('G(x)', fontsize=12)
ax1.set_title('Effect of Mean ($\mu$)\n$\sigma$ = 1.0', fontsize=14)
ax1.legend(fontsize=11)
ax1.set_xlim(-6, 6)

# Effect of standard deviation (sigma)
ax2 = axes[1]
for sigma in [0.5, 1.0, 2.0]:
    y = gaussian_1d(x, mu=0, sigma=sigma)
    ax2.plot(x, y, linewidth=2, label=f'$\sigma$ = {sigma}')
ax2.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
ax2.set_xlabel('x', fontsize=12)
ax2.set_ylabel('G(x)', fontsize=12)
ax2.set_title('Effect of Standard Deviation ($\sigma$)\n$\mu$ = 0', fontsize=14)
ax2.legend(fontsize=11)
ax2.set_xlim(-6, 6)

plt.tight_layout()
plt.show()

print("Key observations:")
print("- Mean (μ) shifts the center of the distribution")
print("- Standard deviation (σ) controls the width/spread")
print("- Smaller σ = taller, narrower peak")
print("- Larger σ = shorter, wider peak")

### Interactive 1D Gaussian

In [ ]:
# Interactive version with sliders
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    
    output = widgets.Output()
    
    mu_slider = widgets.FloatSlider(value=0, min=-4, max=4, step=0.1, description='Mean (μ):')
    sigma_slider = widgets.FloatSlider(value=1, min=0.2, max=3, step=0.1, description='Std Dev (σ):')
    
    def update_plot(mu, sigma):
        with output:
            clear_output(wait=True)
            x = np.linspace(-8, 8, 500)
            y = gaussian_1d(x, mu, sigma)
            
            fig, ax = plt.subplots(figsize=(10, 5))
            ax.plot(x, y, 'b-', linewidth=2)
            ax.fill_between(x, y, alpha=0.3)
            ax.axvline(x=mu, color='red', linestyle='--', label=f'Mean: {mu}')
            ax.axvline(x=mu-sigma, color='green', linestyle=':', alpha=0.7)
            ax.axvline(x=mu+sigma, color='green', linestyle=':', alpha=0.7, label=f'±1σ: [{mu-sigma:.1f}, {mu+sigma:.1f}]')
            
            ax.set_xlim(-8, 8)
            ax.set_ylim(0, 1)
            ax.set_xlabel('x')
            ax.set_ylabel('G(x)')
            ax.set_title(f'1D Gaussian: μ={mu:.1f}, σ={sigma:.1f}')
            ax.legend()
            plt.show()
    
    widgets.interactive(update_plot, mu=mu_slider, sigma=sigma_slider)
    
    display(widgets.VBox([mu_slider, sigma_slider, output]))
    update_plot(0, 1)
    
except ImportError:
    print("ipywidgets not available. Showing static plot instead.")

## 3. 2D Gaussian Distribution

Now let's extend to 2D. This is where things get more interesting!

### Mathematical Definition

A 2D Gaussian is defined as:

$$G(\mathbf{x}) = \frac{1}{2\pi|\Sigma|^{1/2}} \exp\left(-\frac{1}{2}(\mathbf{x}-\boldsymbol{\mu})^T\Sigma^{-1}(\mathbf{x}-\boldsymbol{\mu})\right)$$

Where:
- $\mathbf{x} = [x, y]^T$ - 2D position
- $\boldsymbol{\mu} = [\mu_x, \mu_y]^T$ - 2D mean (center)
- $\Sigma$ - 2×2 **covariance matrix**

### The Covariance Matrix

The covariance matrix $\Sigma$ for 2D is:

$$\Sigma = \begin{bmatrix} \sigma_x^2 & \rho\sigma_x\sigma_y \\ \rho\sigma_x\sigma_y & \sigma_y^2 \end{bmatrix}$$

Where:
- $\sigma_x^2$ = variance in x direction
- $\sigma_y^2$ = variance in y direction
- $\rho$ = correlation coefficient (-1 to 1)

In [ ]:
def gaussian_2d(x, y, mu, cov):
    """
    Evaluate 2D Gaussian at given coordinates.
    
    Args:
        x, y: Coordinate arrays (can be meshgrid)
        mu: Mean [2]
        cov: Covariance matrix [2, 2]
    
    Returns:
        Gaussian values
    """
    mu = np.array(mu)
    cov = np.array(cov)
    
    # Stack coordinates
    pos = np.dstack((x, y))
    
    # Compute inverse and determinant of covariance
    cov_inv = np.linalg.inv(cov)
    cov_det = np.linalg.det(cov)
    
    # Compute Gaussian
    diff = pos - mu
    # Mahalanobis distance: (x-μ)^T Σ^(-1) (x-μ)
    mahal = np.einsum('...i,ij,...j->...', diff, cov_inv, diff)
    
    coefficient = 1 / (2 * np.pi * np.sqrt(cov_det))
    return coefficient * np.exp(-0.5 * mahal)

# Create visualization
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Different covariance configurations
configs = [
    {'name': 'Isotropic (Circle)', 'cov': [[1, 0], [0, 1]]},
    {'name': 'Anisotropic (Ellipse)', 'cov': [[2, 0], [0, 0.5]]},
    {'name': 'Positive Correlation', 'cov': [[1, 0.7], [0.7, 1]]},
    {'name': 'Negative Correlation', 'cov': [[1, -0.7], [-0.7, 1]]},
    {'name': 'Rotated Ellipse', 'cov': [[1.5, 0.8], [0.8, 0.8]]},
    {'name': 'Narrow Ellipse', 'cov': [[3, 0], [0, 0.2]]},
]

x = np.linspace(-4, 4, 200)
y = np.linspace(-4, 4, 200)
X, Y = np.meshgrid(x, y)
mu = [0, 0]

for ax, config in zip(axes.flat, configs):
    cov = config['cov']
    Z = gaussian_2d(X, Y, mu, cov)
    
    # Plot heatmap
    im = ax.imshow(Z, extent=[-4, 4, -4, 4], origin='lower', cmap='hot')
    ax.contour(X, Y, Z, levels=5, colors='white', alpha=0.5)
    
    ax.set_title(f"{config['name']}\nΣ = {cov}", fontsize=10)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

print("Key observations:")
print("- Diagonal covariance (no correlation) → axis-aligned ellipse")
print("- Off-diagonal terms → rotated ellipse")
print("- Positive correlation → tilted upper-right")
print("- Negative correlation → tilted upper-left")

### Understanding Covariance Matrix

The covariance matrix determines the **shape** and **orientation** of the Gaussian ellipse.

Let's visualize this more clearly:

In [ ]:
def plot_2d_gaussian_ellipse(ax, mu, cov, n_std=2, color='blue', alpha=0.3, label=None):
    """
    Plot a 2D Gaussian as an ellipse.
    
    Args:
        ax: Matplotlib axes
        mu: Mean [2]
        cov: Covariance [2, 2]
        n_std: Number of standard deviations
        color: Ellipse color
        alpha: Transparency
    """
    mu = np.array(mu)
    cov = np.array(cov)
    
    # Eigendecomposition
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    
    # Compute angle
    angle = np.degrees(np.arctan2(eigenvectors[1, 1], eigenvectors[0, 1]))
    
    # Compute width and height
    width = 2 * n_std * np.sqrt(eigenvalues[1])
    height = 2 * n_std * np.sqrt(eigenvalues[0])
    
    # Create ellipse
    ellipse = Ellipse(xy=mu, width=width, height=height, angle=angle,
                      facecolor=color, edgecolor=color, alpha=alpha, label=label)
    ax.add_patch(ellipse)
    
    # Plot center
    ax.plot(*mu, 'o', color=color, markersize=8)
    
    # Plot principal axes
    for i in range(2):
        direction = eigenvectors[:, i] * np.sqrt(eigenvalues[i]) * n_std
        ax.arrow(mu[0], mu[1], direction[0], direction[1],
                head_width=0.1, head_length=0.05, fc=color, ec=color)
    
    return ellipse

# Demonstrate
fig, ax = plt.subplots(figsize=(10, 10))

# Multiple Gaussians
gaussians = [
    {'mu': [0, 0], 'cov': [[1, 0], [0, 1]], 'color': 'blue', 'label': 'Isotropic'},
    {'mu': [2, 2], 'cov': [[2, 0.8], [0.8, 0.5]], 'color': 'red', 'label': 'Anisotropic'},
    {'mu': [-2, 2], 'cov': [[0.5, -0.4], [-0.4, 1.5]], 'color': 'green', 'label': 'Correlated'},
    {'mu': [-2, -2], 'cov': [[1.5, 0], [0, 0.3]], 'color': 'orange', 'label': 'Axis-aligned'},
]

for g in gaussians:
    plot_2d_gaussian_ellipse(ax, g['mu'], g['cov'], n_std=2, 
                              color=g['color'], alpha=0.4, label=g['label'])

ax.set_xlim(-5, 5)
ax.set_ylim(-5, 5)
ax.set_aspect('equal')
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title('2D Gaussian Ellipses (2σ contours)\nArrows show principal axes', fontsize=14)
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

plt.show()

print("\nThe arrows show the eigenvectors (principal axes) of the covariance matrix.")
print("Their lengths are proportional to the square root of eigenvalues (standard deviations).")

### Interactive 2D Gaussian

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    
    output2d = widgets.Output()
    
    var_x = widgets.FloatSlider(value=1, min=0.1, max=3, step=0.1, description='Var X:')
    var_y = widgets.FloatSlider(value=1, min=0.1, max=3, step=0.1, description='Var Y:')
    corr = widgets.FloatSlider(value=0, min=-0.9, max=0.9, step=0.1, description='Correlation:')
    
    def update_2d(var_x_val, var_y_val, corr_val):
        with output2d:
            clear_output(wait=True)
            
            # Build covariance matrix
            cov_xy = corr_val * np.sqrt(var_x_val * var_y_val)
            cov = np.array([[var_x_val, cov_xy], [cov_xy, var_y_val]])
            
            fig, axes = plt.subplots(1, 2, figsize=(14, 6))
            
            # Heatmap
            x = np.linspace(-4, 4, 200)
            y = np.linspace(-4, 4, 200)
            X, Y = np.meshgrid(x, y)
            Z = gaussian_2d(X, Y, [0, 0], cov)
            
            axes[0].imshow(Z, extent=[-4, 4, -4, 4], origin='lower', cmap='viridis')
            axes[0].contour(X, Y, Z, levels=5, colors='white', alpha=0.6)
            axes[0].set_title('Gaussian Heatmap')
            axes[0].set_xlabel('x')
            axes[0].set_ylabel('y')
            
            # Ellipse
            plot_2d_gaussian_ellipse(axes[1], [0, 0], cov, n_std=2, color='blue', alpha=0.4)
            axes[1].set_xlim(-4, 4)
            axes[1].set_ylim(-4, 4)
            axes[1].set_aspect('equal')
            axes[1].set_title('2σ Ellipse with Principal Axes')
            axes[1].set_xlabel('x')
            axes[1].set_ylabel('y')
            axes[1].grid(True, alpha=0.3)
            
            plt.suptitle(f'Covariance Matrix: [[{var_x_val:.1f}, {cov_xy:.2f}], [{cov_xy:.2f}, {var_y_val:.1f}]]')
            plt.tight_layout()
            plt.show()
    
    widgets.interactive(update_2d, var_x_val=var_x, var_y_val=var_y, corr_val=corr)
    display(widgets.VBox([var_x, var_y, corr, output2d]))
    update_2d(1, 1, 0)
    
except ImportError:
    print("Interactive widgets not available.")

## 4. 3D Gaussian Distribution

Now we're ready for 3D Gaussians - the building blocks of 3D Gaussian Splatting!

### Mathematical Definition

A 3D Gaussian is defined as:

$$G(\mathbf{x}) = \exp\left(-\frac{1}{2}(\mathbf{x}-\boldsymbol{\mu})^T\Sigma^{-1}(\mathbf{x}-\boldsymbol{\mu})\right)$$

Where:
- $\mathbf{x} = [x, y, z]^T$ - 3D position
- $\boldsymbol{\mu} = [\mu_x, \mu_y, \mu_z]^T$ - 3D mean (center)
- $\Sigma$ - 3×3 covariance matrix (symmetric, positive definite)

**Note**: In 3DGS, we often drop the normalization constant since we care about relative values.

In [ ]:
def plot_3d_ellipsoid(ax, mu, cov, n_std=2, color='blue', alpha=0.3, resolution=20):
    """
    Plot a 3D Gaussian as an ellipsoid.
    
    Args:
        ax: Matplotlib 3D axes
        mu: Mean [3]
        cov: Covariance [3, 3]
        n_std: Number of standard deviations
        color: Surface color
        alpha: Transparency
        resolution: Surface resolution
    """
    mu = np.array(mu)
    cov = np.array(cov)
    
    # Eigendecomposition
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    
    # Create unit sphere
    u = np.linspace(0, 2 * np.pi, resolution)
    v = np.linspace(0, np.pi, resolution)
    x = np.outer(np.cos(u), np.sin(v))
    y = np.outer(np.sin(u), np.sin(v))
    z = np.outer(np.ones_like(u), np.cos(v))
    
    # Scale by eigenvalues (std devs) and rotate by eigenvectors
    radii = n_std * np.sqrt(eigenvalues)
    sphere_points = np.stack([x, y, z], axis=-1)  # [res, res, 3]
    
    # Transform: rotate and scale
    ellipsoid_points = np.einsum(
        'ij,klj->kli',
        eigenvectors @ np.diag(radii),
        sphere_points
    ) + mu
    
    # Plot surface
    ax.plot_surface(
        ellipsoid_points[:, :, 0],
        ellipsoid_points[:, :, 1],
        ellipsoid_points[:, :, 2],
        color=color, alpha=alpha, shade=True
    )
    
    # Plot center
    ax.scatter(*mu, color=color, s=50)
    
    return ellipsoid_points

# Create 3D visualization
fig = plt.figure(figsize=(16, 5))

# Example 1: Isotropic (sphere)
ax1 = fig.add_subplot(131, projection='3d')
cov1 = np.eye(3)
plot_3d_ellipsoid(ax1, [0, 0, 0], cov1, n_std=2, color='blue', alpha=0.4)
ax1.set_title('Isotropic (Sphere)\nΣ = I')
ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_zlabel('Z')

# Example 2: Anisotropic (elongated)
ax2 = fig.add_subplot(132, projection='3d')
cov2 = np.diag([2, 0.5, 0.3])
plot_3d_ellipsoid(ax2, [0, 0, 0], cov2, n_std=2, color='red', alpha=0.4)
ax2.set_title('Anisotropic\nΣ = diag(2, 0.5, 0.3)')
ax2.set_xlabel('X')
ax2.set_ylabel('Y')
ax2.set_zlabel('Z')

# Example 3: Rotated
ax3 = fig.add_subplot(133, projection='3d')
# Create rotated covariance
angle = np.pi / 4  # 45 degrees
R = np.array([
    [np.cos(angle), -np.sin(angle), 0],
    [np.sin(angle), np.cos(angle), 0],
    [0, 0, 1]
])
S = np.diag([2, 0.5, 0.3])
cov3 = R @ S @ S.T @ R.T
plot_3d_ellipsoid(ax3, [0, 0, 0], cov3, n_std=2, color='green', alpha=0.4)
ax3.set_title('Rotated Anisotropic\nΣ = R @ S @ Sᵀ @ Rᵀ')
ax3.set_xlabel('X')
ax3.set_ylabel('Y')
ax3.set_zlabel('Z')

plt.tight_layout()
plt.show()

print("\nIn 3D Gaussian Splatting:")
print("- Each 'splat' is a 3D Gaussian ellipsoid")
print("- The shape is controlled by the covariance matrix")
print("- We parameterize it as Σ = R @ S @ Sᵀ @ Rᵀ")
print("  where R = rotation matrix, S = diagonal scaling matrix")

## 5. Covariance Decomposition (Key for 3DGS!)

In 3D Gaussian Splatting, we don't directly optimize the covariance matrix. Instead, we decompose it as:

$$\Sigma = R \cdot S \cdot S^T \cdot R^T$$

Where:
- $R$ = rotation matrix (from quaternion $q = [w, x, y, z]$)
- $S$ = diagonal scaling matrix $\text{diag}(s_x, s_y, s_z)$

This ensures:
1. The covariance is always positive semi-definite
2. We can directly optimize scale and rotation
3. The parameters have intuitive physical meaning

In [ ]:
def quaternion_to_rotation_matrix(q):
    """
    Convert quaternion [w, x, y, z] to rotation matrix.
    """
    w, x, y, z = q / np.linalg.norm(q)  # Normalize
    
    R = np.array([
        [1 - 2*(y*y + z*z), 2*(x*y - w*z), 2*(x*z + w*y)],
        [2*(x*y + w*z), 1 - 2*(x*x + z*z), 2*(y*z - w*x)],
        [2*(x*z - w*y), 2*(y*z + w*x), 1 - 2*(x*x + y*y)]
    ])
    
    return R

def build_covariance(scaling, rotation_quat):
    """
    Build covariance matrix from scaling and rotation.
    
    Args:
        scaling: [sx, sy, sz] scale factors
        rotation_quat: [w, x, y, z] quaternion
    
    Returns:
        Covariance matrix [3, 3]
    """
    S = np.diag(scaling)
    R = quaternion_to_rotation_matrix(rotation_quat)
    
    # Σ = R @ S @ S.T @ R.T
    cov = R @ S @ S.T @ R.T
    
    return cov

# Demonstrate the decomposition
print("Covariance Matrix Decomposition")
print("=" * 50)

# Define parameters
scaling = np.array([1.5, 0.5, 0.3])  # sx, sy, sz
rotation_quat = np.array([0.924, 0.0, 0.383, 0.0])  # ~45 degree rotation around Y

print(f"\nScaling (S): {scaling}")
print(f"Rotation quaternion (q): {rotation_quat}")

# Build rotation matrix
R = quaternion_to_rotation_matrix(rotation_quat)
print(f"\nRotation matrix (R):\n{R.round(3)}")

# Build covariance
cov = build_covariance(scaling, rotation_quat)
print(f"\nCovariance matrix (Σ = R @ S @ S.T @ R.T):\n{cov.round(3)}")

# Verify: eigendecomposition should give back our scaling
eigenvalues, eigenvectors = np.linalg.eigh(cov)
print(f"\nVerification - Eigenvalues: {np.sqrt(eigenvalues).round(3)}")
print(f"(Should approximately match sorted scaling: {np.sort(scaling)}")

In [ ]:
# Visualize the effect of scaling and rotation
fig = plt.figure(figsize=(16, 5))

# Original orientation
ax1 = fig.add_subplot(131, projection='3d')
scaling1 = [1.5, 0.5, 0.3]
quat1 = [1, 0, 0, 0]  # Identity (no rotation)
cov1 = build_covariance(scaling1, quat1)
plot_3d_ellipsoid(ax1, [0, 0, 0], cov1, n_std=2, color='blue', alpha=0.4)
ax1.set_title(f'No rotation\nq = {quat1}')
ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_zlabel('Z')

# Rotated around Z
ax2 = fig.add_subplot(132, projection='3d')
angle_z = np.pi / 4
quat2 = [np.cos(angle_z/2), 0, 0, np.sin(angle_z/2)]  # 45° around Z
cov2 = build_covariance(scaling1, quat2)
plot_3d_ellipsoid(ax2, [0, 0, 0], cov2, n_std=2, color='red', alpha=0.4)
ax2.set_title(f'45° around Z\nq = [{quat2[0]:.2f}, 0, 0, {quat2[3]:.2f}]')
ax2.set_xlabel('X')
ax2.set_ylabel('Y')
ax2.set_zlabel('Z')

# Rotated around Y
ax3 = fig.add_subplot(133, projection='3d')
angle_y = np.pi / 3
quat3 = [np.cos(angle_y/2), 0, np.sin(angle_y/2), 0]  # 60° around Y
cov3 = build_covariance(scaling1, quat3)
plot_3d_ellipsoid(ax3, [0, 0, 0], cov3, n_std=2, color='green', alpha=0.4)
ax3.set_title(f'60° around Y\nq = [{quat3[0]:.2f}, 0, {quat3[2]:.2f}, 0]')
ax3.set_xlabel('X')
ax3.set_ylabel('Y')
ax3.set_zlabel('Z')

# Set same limits for all
for ax in [ax1, ax2, ax3]:
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.set_zlim(-3, 3)

plt.tight_layout()
plt.show()

print("\nThe same scaling [1.5, 0.5, 0.3] with different rotations")
print("produces ellipsoids with the same shape but different orientations.")

## 6. Summary: Gaussian Parameters in 3DGS

In 3D Gaussian Splatting, each Gaussian primitive is defined by:

| Parameter | Symbol | Dimension | Description |
|-----------|--------|-----------|-------------|
| **Position** | $\boldsymbol{\mu}$ | 3 | Center of the Gaussian $(x, y, z)$ |
| **Scaling** | $\mathbf{s}$ | 3 | Size along each axis $(s_x, s_y, s_z)$ |
| **Rotation** | $\mathbf{q}$ | 4 | Orientation as quaternion $(w, x, y, z)$ |
| **Opacity** | $\alpha$ | 1 | Transparency (0-1) |
| **Color/SH** | $\mathbf{c}$ | 3-48 | RGB or Spherical Harmonics coefficients |

**Total parameters per Gaussian**: ~59 (with degree-3 SH)

In [ ]:
# Summary visualization: A scene with multiple Gaussians
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

# Create a simple "scene" with multiple Gaussians
gaussians = [
    {'mu': [0, 0, 0], 's': [0.8, 0.3, 0.2], 'q': [1, 0, 0, 0], 'color': 'red'},
    {'mu': [2, 0, 0], 's': [0.3, 0.3, 0.8], 'q': [1, 0, 0, 0], 'color': 'green'},
    {'mu': [0, 2, 0], 's': [0.5, 0.5, 0.5], 'q': [1, 0, 0, 0], 'color': 'blue'},
    {'mu': [1, 1, 1], 's': [0.4, 0.2, 0.6], 'q': [0.924, 0.383, 0, 0], 'color': 'orange'},
    {'mu': [-1, 0.5, 0.5], 's': [0.3, 0.7, 0.3], 'q': [0.707, 0, 0.707, 0], 'color': 'purple'},
]

for g in gaussians:
    cov = build_covariance(g['s'], g['q'])
    plot_3d_ellipsoid(ax, g['mu'], cov, n_std=2, color=g['color'], alpha=0.5)

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('A Simple "Scene" with 5 Gaussian Primitives\n(This is the foundation of 3D Gaussian Splatting!)', fontsize=14)

ax.set_xlim(-3, 4)
ax.set_ylim(-2, 4)
ax.set_zlim(-2, 3)

plt.tight_layout()
plt.show()

print("\nThis is essentially what 3DGS does:")
print("1. Represent a scene with many 3D Gaussians")
print("2. Each Gaussian has position, size, orientation, color, and opacity")
print("3. Optimize these parameters to match input images")
print("4. Render by 'splatting' Gaussians onto the image plane")

## Key Takeaways

1. **Gaussians are smooth, continuous distributions** - perfect for rendering
2. **1D to 3D**: Mean controls position, covariance controls shape
3. **Covariance decomposition**: $\Sigma = RSS^TR^T$ ensures valid covariance
4. **Quaternions for rotation**: Compact, singularity-free representation
5. **3DGS primitives**: Position + Scale + Rotation + Opacity + Color

---

## Next Steps

In the next notebook, we'll dive deeper into:
- **3D Gaussian Math**: Formal treatment of the covariance decomposition
- **PyTorch implementation**: How to implement these concepts in code

**[02_3d_gaussian_math.ipynb](./02_3d_gaussian_math.ipynb)** - 3D Gaussian Mathematics